# Roch Stability Criterion — All Density Pathways

This notebook applies the **Roch natural stability criterion** to all ECTP slabs across all four
density calculation pathways.  The computation is entirely independent of the WEAC
solver — only density and slope angle are required.

## What is the Roch criterion?

> **Natural variant** (Roch 1966):
> $$S_r = \frac{\tau_c}{\tau}, \quad \tau = \sum_i \rho_i h_i g \sin\theta$$
>
> Values below 1 indicate the weak layer is **potentially unstable**.

> **Skier variant** (Föhn 1987, optional):
> $$S_a = \frac{\tau_c}{\tau + \delta\tau}$$
>
> where $\delta\tau$ is the skier-induced shear stress at the weak layer, supplied by the
> caller (not computed automatically).  Uncomment `TAU_SK = 150.0` in the params cell
> and the S_a block in the run-roch cell to enable.

## Why 4 pathways (not 32)?

Roch requires only **density** for each slab layer — no elastic modulus, shear modulus,
or Poisson's ratio.  The four density methods give four independent pathways:

| Method | Inputs required |
|---|---|
| `data_flow` | Measured density (direct snow-pit observation) |
| `geldsetzer` | Hand hardness + grain form (Geldsetzer & Jamieson 2000) |
| `kim_jamieson_table2` | Hand hardness + grain form (Kim & Jamieson 2006, Table 2) |
| `kim_jamieson_table5` | Hand hardness + grain form + grain size (Kim & Jamieson 2006, Table 5) |

## Table of Contents

1. [Setup & Imports](#1-setup--imports)
2. [Load ECTP Slabs](#2-load-ectp-slabs)
3. [Pathways Overview](#3-pathways-overview)
4. [Criterion Parameters](#4-criterion-parameters)
5. [Run Roch — All Slabs](#5-run-roch--all-slabs)
6. [Summary Table](#6-summary-table)
7. [S_r Distribution by Density Pathway (Natural Variant)](#7-s_r-distribution-by-density-pathway-natural-variant)
8. [Skier Variant (S_a) — Optional](#8-skier-variant-s_a--optional)

## 1. Setup & Imports

In [1]:
import warnings
warnings.filterwarnings('ignore')

import math
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from tqdm import tqdm

from snowpyt_mechparams.graph import graph
from snowpyt_mechparams.algorithm import find_parameterizations
from snowpyt_mechparams.execution import ExecutionEngine
from snowpyt_mechparams.execution.config import ExecutionConfig
from snowpyt_mechparams.stability_criteria.roch import calculate_roch

from notebook_utils import (
    load_pits, create_ectp_slabs,
    DENSITY_COLORS, rgba,
)

print('Imports OK')

Imports OK


## 2. Load ECTP Slabs

In [2]:
pits = load_pits()
print(f'Loaded {len(pits)} snow pits ({sum(len(pit.layers) for pit in pits):,} layers)')

ectp_slabs = create_ectp_slabs(pits)
print(f'Created {len(ectp_slabs):,} ECTP slabs')

Loaded 50278 snow pits (371,429 layers)
Created 14,776 ECTP slabs


## 3. Pathways Overview

Enumerate the density methods from the dependency graph — these are the only free
variables for the Roch criterion.

In [3]:
pathways = find_parameterizations(graph, graph.get_node('density'))

print(f'Found {len(pathways)} pathways for calculating density:\n')
for i, pathway in enumerate(pathways, 1):
    print(f'Pathway {i}:')
    print(pathway)
    print()

# All four density methods (data_flow = direct cutter measurement)
DENSITY_METHODS = ['data_flow', 'geldsetzer', 'kim_jamieson_table2', 'kim_jamieson_table5']

DENSITY_DESCRIPTIONS = {
    'data_flow':           'Measured density (direct snow-pit observation)',
    'geldsetzer':          'Hand hardness + grain form (Geldsetzer & Jamieson 2000)',
    'kim_jamieson_table2': 'Hand hardness + grain form (Kim & Jamieson 2006, Table 2)',
    'kim_jamieson_table5': 'Hand hardness + grain form + grain size (Kim & Jamieson 2006, Table 5)',
}

print(f'{len(DENSITY_METHODS)} density methods → {len(DENSITY_METHODS)} Roch pathways (one per density method):')
for m in DENSITY_METHODS:
    print(f'  {m:<30s}  {DENSITY_DESCRIPTIONS.get(m, "")}')
print()
print('Compare: WEAC criterion has 32 pathways (density × E-mod × ν × weak-layer fracture params).')
print('Roch needs only density and slope angle — no elastic or fracture parameters.')

Found 4 pathways for calculating density:

Pathway 1:
branch 1: snow_pit -- data_flow --> measured_density -- data_flow --> density

Pathway 2:
branch 1: snow_pit -- data_flow --> measured_hand_hardness -- data_flow --> merge_hand_hardness_grain_form
branch 2: snow_pit -- data_flow --> measured_grain_form -- data_flow --> merge_hand_hardness_grain_form
merge branch 1, branch 2: merge_hand_hardness_grain_form -- geldsetzer --> density

Pathway 3:
branch 1: snow_pit -- data_flow --> measured_hand_hardness -- data_flow --> merge_hand_hardness_grain_form
branch 2: snow_pit -- data_flow --> measured_grain_form -- data_flow --> merge_hand_hardness_grain_form
merge branch 1, branch 2: merge_hand_hardness_grain_form -- kim_jamieson_table2 --> density

Pathway 4:
branch 1: snow_pit -- data_flow --> measured_hand_hardness -- data_flow --> merge_hand_hardness_grain_form
branch 2: snow_pit -- data_flow --> measured_grain_form -- data_flow --> merge_hand_hardness_grain_form
branch 3: snow_pit -- da

## 4. Criterion Parameters

- **τ_c (shear strength)**: weak-layer shear strength in N/m².  This should ideally come
  from direct shear measurements; a representative literature value for faceted-crystal
  weak layers is used here (Schweizer & Jamieson 2003).  Adjust `TAU_C` to explore
  sensitivity.

- **τ_sk (skier stress)**: additional shear stress at weak-layer depth from a skier
  (Föhn 1987 reference value).

In [4]:
# Weak-layer shear strength [N/m²].
# Representative value for faceted crystals / depth hoar (Schweizer & Jamieson 2003).
# Typical range for alpine weak layers: 200–2000 N/m² depending on grain form and age.
TAU_C = 500.0   # N/m²

# To also compute the skier variant S_a = τ_c / (τ + δτ), pass skier_stress=TAU_SK
# to calculate_roch. Uncomment the line below and the S_a block in the run-roch cell.
# TAU_SK = 150.0  # N/m² — δτ beneath a skier on a 25° slope (Föhn 1987 reference value)

print(f'τ_c  (shear strength, Schweizer & Jamieson 2003 reference): {TAU_C:.0f} N/m²')

τ_c  (shear strength, Schweizer & Jamieson 2003 reference): 500 N/m²


## 5. Run Roch — All Slabs

The execution engine targets `'density'` directly, returning exactly one pathway per
density method (4 pathways total).  The Roch natural criterion
$S_r = \tau_c / \tau$ is computed for every slab with a valid density.

> **Skier variant:** uncomment the `TAU_SK` line in the params cell and the S_a block
> below to additionally compute $S_a = \tau_c / (\tau + \delta\tau)$ with
> $\delta\tau = 150$ N/m² (Föhn 1987 reference value).

In [5]:
engine = ExecutionEngine(graph)
config = ExecutionConfig(include_method_uncertainty=False)
total  = len(ectp_slabs)

# Accumulators per density pathway
pathway_S_r  = {m: [] for m in DENSITY_METHODS}  # natural variant: S_r = τ_c / τ
pathway_tau  = {m: [] for m in DENSITY_METHODS}  # gravitational shear stress τ [N/m²]

# Skier variant accumulator — uncomment to enable (also uncomment TAU_SK in params cell)
# pathway_S_a = {m: [] for m in DENSITY_METHODS}  # S_a = τ_c / (τ + δτ)

for slab in tqdm(ectp_slabs, desc='Running Roch', unit='slab'):
    n    = len(slab.layers)

    # Target 'density' directly — 4 pathways (one per density method), no WEAC involved.
    results = engine.execute_all(slab, 'density', config=config)

    for pr in results.pathways.values():
        density_method = pr.methods_used.get('density', '?')
        if density_method not in DENSITY_METHODS or pr.slab is None:
            continue

        # Only proceed when density was computed for every layer in the slab
        n_ok = sum(
            1 for t in pr.computation_trace
            if t.parameter == 'density' and t.success and t.output is not None
        )
        if n_ok != n:
            continue

        # Natural variant: S_r = τ_c / τ
        roch_nat = calculate_roch(pr.slab, tau_c=TAU_C)
        if roch_nat is not None:
            S_r   = float(getattr(roch_nat.stability_index, 'nominal_value', roch_nat.stability_index))
            tau_v = float(getattr(roch_nat.shear_stress,    'nominal_value', roch_nat.shear_stress))
            if math.isfinite(S_r):
                pathway_S_r[density_method].append(S_r)
                pathway_tau[density_method].append(tau_v)

        # Skier variant: S_a = τ_c / (τ + δτ) — uncomment to enable
        # roch_sk = calculate_roch(pr.slab, tau_c=TAU_C, skier_stress=TAU_SK)
        # if roch_sk is not None:
        #     S_a = float(getattr(roch_sk.stability_index, 'nominal_value', roch_sk.stability_index))
        #     if math.isfinite(S_a):
        #         pathway_S_a[density_method].append(S_a)

# Sort by S_r coverage (most successful first) for all subsequent cells
sorted_methods = sorted(DENSITY_METHODS, key=lambda m: -len(pathway_S_r[m]))

print(f'Done.  {sum(len(v) for v in pathway_S_r.values())} slab/pathway combinations yielded S_r.')

Running Roch: 100%|██████████| 14776/14776 [00:03<00:00, 4435.23slab/s]

Done.  10841 slab/pathway combinations yielded S_r.


## 6. Summary Table

In [6]:
rows = []
for m in sorted_methods:
    Sr  = np.array(pathway_S_r[m])
    tau = np.array(pathway_tau[m])
    n   = len(Sr)
    rows.append({
        'Density method':   m,
        'N (S_r)':          n,
        'Coverage':         f'{n / total:.1%}',
        'Median τ (N/m²)':  round(float(np.median(tau)), 1)    if n else float('nan'),
        'Median S_r':       round(float(np.median(Sr)),  3)    if n else float('nan'),
        '% S_r < 1':        f'{100 * np.mean(Sr < 1.0):.1f}%' if n else 'N/A',
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))
print()
print(f'Total ECTP slabs : {total}')
print(f'τ_c              : {TAU_C:.0f} N/m²  (Schweizer & Jamieson 2003 reference)')

     Density method  N (S_r) Coverage  Median τ (N/m²)  Median S_r % S_r < 1
kim_jamieson_table2     5475    37.1%            262.8    1902.799      0.0%
         geldsetzer     4200    28.4%            246.3    2030.071      0.0%
kim_jamieson_table5     1066     7.2%            195.9    2552.604      0.0%
          data_flow      100     0.7%            249.6    2003.394      0.0%

Total ECTP slabs : 14776
τ_c              : 500 N/m²  (Schweizer & Jamieson 2003 reference)


## 7. S_r Distribution by Density Pathway (Natural Variant)

Violin plots of the Roch natural stability index $S_r = \tau_c / \tau$ across all ECTP
slabs, ordered by coverage (highest coverage at top).  The red dashed line at $S_r = 1$
marks the instability threshold.

In [7]:
# DENSITY_COLORS imported from notebook_utils (Wong 2011 palette)

fig_nat = go.Figure()

for m in reversed(sorted_methods):   # reversed so highest-coverage ends up at top
    vals = np.array(pathway_S_r[m])
    if len(vals) == 0:
        continue
    n          = len(vals)
    pct        = n / total
    color      = rgba(DENSITY_COLORS.get(m, '#888888'), 0.75)
    line_color = rgba(DENSITY_COLORS.get(m, '#888888'), 1.0)

    fig_nat.add_trace(go.Violin(
        x=vals,
        y=[f"{m}  <i>({n:,} / {total:,}, {pct:.1%})</i>"] * n,
        orientation='h',
        name=m,
        box_visible=True,
        meanline_visible=True,
        points=False,
        fillcolor=color,
        line_color=line_color,
        opacity=0.85,
        showlegend=False,
    ))

fig_nat.add_vline(
    x=1.0,
    line_dash='dash',
    line_color='rgba(196,84,78,0.85)',
    line_width=2,
    annotation_text='S_r = 1  (instability threshold)',
    annotation_position='top right',
)

fig_nat.update_layout(
    title=dict(
        text=(
            '<b>Roch Natural Stability Index S_r by Density Pathway</b><br>'
            f'<sup>ECTP slabs — ordered top→bottom by coverage — '
            f'box = IQR, line = mean — τ_c = {TAU_C:.0f} N/m²</sup>'
        ),
        x=0.5, xanchor='center', font=dict(size=13),
    ),
    xaxis=dict(
        title='S_r  (τ_c / τ)',
        gridcolor='rgba(200,200,200,0.4)',
        zeroline=True,
        zerolinecolor='rgba(180,180,180,0.5)',
    ),
    yaxis=dict(
        autorange='reversed',
        tickfont=dict(size=10),
    ),
    plot_bgcolor='white',
    violingap=0.08,
    violingroupgap=0,
    width=950,
    height=420,
    margin=dict(l=340, r=40, t=90, b=60),
)

fig_nat.show()

## 8. Skier Variant (S_a) — Optional

To compute $S_a = \tau_c / (\tau + \delta\tau)$ with $\delta\tau = 150$ N/m²
(Föhn 1987 reference value):

1. Uncomment `TAU_SK = 150.0` in **Cell 4 (Criterion Parameters)**
2. Uncomment the S_a block in **Cell 5 (Run Roch)**
3. Re-run those cells — `pathway_S_a` will be populated with the skier stability index

The `plot-ssk` cell below contains a ready-to-use violin plot for S_a; uncomment it after
enabling `pathway_S_a`.

In [8]:
# Skier variant violin plot — uncomment after enabling pathway_S_a in the run-roch cell.
#
# fig_sk = go.Figure()
# for m in reversed(sorted_methods):
#     vals = np.array(pathway_S_a[m])
#     if len(vals) == 0:
#         continue
#     n          = len(vals)
#     pct        = n / total
#     color      = rgba(DENSITY_COLORS.get(m, '#888888'), 0.75)
#     line_color = rgba(DENSITY_COLORS.get(m, '#888888'), 1.0)
#     fig_sk.add_trace(go.Violin(
#         x=vals,
#         y=[f"{m}  <i>({n:,} / {total:,}, {pct:.1%})</i>"] * n,
#         orientation='h', name=m,
#         box_visible=True, meanline_visible=True, points=False,
#         fillcolor=color, line_color=line_color, opacity=0.85, showlegend=False,
#     ))
# fig_sk.add_vline(
#     x=1.0, line_dash='dash', line_color='rgba(196,84,78,0.85)', line_width=2,
#     annotation_text=f'S_a = 1  (skier triggers instability, δτ = {TAU_SK:.0f} N/m²)',
#     annotation_position='top right',
# )
# fig_sk.update_layout(
#     title=dict(
#         text=(
#             '<b>Roch Skier Stability Index S_a by Density Pathway</b><br>'
#             f'<sup>S_a = τ_c / (τ + δτ),  δτ = {TAU_SK:.0f} N/m² (Föhn 1987)</sup>'
#         ),
#         x=0.5, xanchor='center', font=dict(size=13),
#     ),
#     xaxis=dict(title='S_a  (< 1 → unstable)', gridcolor='rgba(200,200,200,0.4)'),
#     yaxis=dict(autorange='reversed', tickfont=dict(size=10)),
#     plot_bgcolor='white', violingap=0.08,
#     width=950, height=420, margin=dict(l=340, r=40, t=90, b=60),
# )
# fig_sk.show()
print('Skier variant plot disabled — uncomment to enable (requires pathway_S_a).')

Skier variant plot disabled — uncomment to enable (requires pathway_S_a).
